# Corporate Default Prediction
XGBoost classifier predicting US public company bankruptcy from NYSE/NASDAQ accounting data (1999-2018).

**Dataset:** [American Companies Bankruptcy Prediction](https://www.kaggle.com/datasets/utkarshx27/american-companies-bankruptcy-prediction-dataset) - 78,682 firm-year observations from 8,262 companies.

**Approach:** Temporal train/test split, XGBoost with class weighting, SHAP interpretability.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import (
    roc_auc_score, precision_recall_curve, auc,
    confusion_matrix, classification_report, roc_curve
)
import shap
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 30)

## Data Loading

In [ ]:
df = pd.read_csv("data/american_bankruptcy.csv")

COLUMN_MAP = {
    "X1": "current_assets",
    "X2": "cogs",
    "X3": "depreciation_amort",
    "X4": "ebitda",
    "X5": "inventory",
    "X6": "net_income",
    "X7": "total_receivables",
    "X8": "market_cap",
    "X9": "net_sales",
    "X10": "total_assets",
    "X11": "total_lt_debt",
    "X12": "ebit",
    "X13": "gross_profit",
    "X14": "total_current_liabilities",
    "X15": "retained_earnings",
    "X16": "total_revenue",
    "X17": "total_liabilities",
    "X18": "total_operating_expenses",
}
df.rename(columns=COLUMN_MAP, inplace=True)

# Encode target: failed=1, alive=0
df["target"] = (df["status_label"] == "failed").astype(int)

# Drop duplicate column (total_revenue is identical to net_sales) and non-feature columns
df.drop(columns=["total_revenue", "status_label", "company_name"], inplace=True)

print(f"Shape: {df.shape}")
print(f"Default rate: {df['target'].mean():.2%}")
df.head()

## Exploratory Data Analysis

In [ ]:
print(f"Default rate: {df['target'].mean():.2%}")
print(f"\nClass distribution:\n{df['target'].value_counts().rename({0: 'alive', 1: 'failed'})}")

In [ ]:
# Feature distributions split by default status - 6 key financial variables
key_features = ["total_assets", "net_income", "ebitda", "total_lt_debt", "market_cap", "net_sales"]
labels = {0: "alive", 1: "failed"}
colors = {0: "steelblue", 1: "crimson"}

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(key_features):
    ax = axes[i]
    for target_val in [0, 1]:
        subset = df.loc[df["target"] == target_val, col].dropna()
        # Shift to positive domain for log scale (add small offset if needed)
        shift = max(0, -subset.min()) + 1
        ax.hist(
            subset + shift,
            bins=60,
            alpha=0.5,
            color=colors[target_val],
            label=labels[target_val],
            density=True,
        )
    ax.set_xscale("log")
    ax.set_title(col.replace("_", " ").title())
    ax.set_xlabel("Value (log scale, shifted to positive)")
    ax.set_ylabel("Density")
    ax.legend()

fig.suptitle("Feature Distributions by Default Status", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
corr = df.select_dtypes(include=[np.number]).drop(columns=["target", "year"]).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap="RdBu_r", center=0, ax=ax,
            fmt=".1f", linewidths=0.5, cbar_kws={"shrink": 0.8})
ax.set_title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

## Feature Engineering

In [ ]:
df["current_ratio"] = df["current_assets"] / df["total_assets"].replace(0, np.nan)
df["debt_to_assets"] = df["total_lt_debt"] / df["total_assets"].replace(0, np.nan)
df["net_margin"] = df["net_income"] / df["net_sales"].replace(0, np.nan)
df["ebitda_margin"] = df["ebitda"] / df["net_sales"].replace(0, np.nan)
df["roa"] = df["net_income"] / df["total_assets"].replace(0, np.nan)
df["gross_margin"] = df["gross_profit"] / df["net_sales"].replace(0, np.nan)
df["asset_turnover"] = df["net_sales"] / df["total_assets"].replace(0, np.nan)
df["debt_to_ebitda"] = df["total_lt_debt"] / df["ebitda"].replace(0, np.nan)

In [ ]:
feature_cols = [c for c in df.columns if c not in ["target", "year"]]

# Clip at 1st/99th percentile to reduce outlier influence
for col in feature_cols:
    lower, upper = df[col].quantile(0.01), df[col].quantile(0.99)
    df[col] = df[col].clip(lower, upper)

# Fill any NaN introduced by division guards with column median
df[feature_cols] = df[feature_cols].fillna(df[feature_cols].median())

print(f"Features after engineering: {len(feature_cols)}")
print(f"Any remaining NaN: {df[feature_cols].isna().any().any()}")

| Ratio | Formula | Measures |
|-------|---------|----------|
| current_ratio | Current Assets / Total Assets | Liquidity |
| debt_to_assets | Long-Term Debt / Total Assets | Leverage |
| net_margin | Net Income / Net Sales | Profitability |
| ebitda_margin | EBITDA / Net Sales | Operating efficiency |
| roa | Net Income / Total Assets | Return on assets |
| gross_margin | Gross Profit / Net Sales | Pricing power |
| asset_turnover | Net Sales / Total Assets | Asset utilization |
| debt_to_ebitda | Long-Term Debt / EBITDA | Debt serviceability |

## Train / Validation / Test Split

Temporal split - no data leakage across time:
- **Train:** 1999-2011
- **Validation:** 2012-2014
- **Test:** 2015-2018

In [ ]:
feature_cols = [c for c in df.columns if c not in ["target", "year"]]

train = df[df["year"] <= 2011]
val = df[(df["year"] >= 2012) & (df["year"] <= 2014)]
test = df[df["year"] >= 2015]

X_train, y_train = train[feature_cols], train["target"]
X_val, y_val = val[feature_cols], val["target"]
X_test, y_test = test[feature_cols], test["target"]

print(f"Train: {X_train.shape[0]:,} obs ({y_train.mean():.2%} default rate)")
print(f"Val:   {X_val.shape[0]:,} obs ({y_val.mean():.2%} default rate)")
print(f"Test:  {X_test.shape[0]:,} obs ({y_test.mean():.2%} default rate)")